# MGSC 661 Final Project | Part C
## Results visualization and interpretation

Turn Part A's final logistic model into presentation-ready figures, an H1–H5 verdict table, and the numbers behind our recommendations. Run from the repository root (same as Part B). Figures are saved to the `Part C/` folder.

All results are predictive associations, not causal effects.

## 1. Load Part A outputs

**Purpose.** Use Part A's final coefficients, odds-ratio table and centering constants so every number here matches the main model. `predict()` turns the coefficients into a predicted success probability for any campaign profile.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

part_a = Path("Part A") if Path("Part A").is_dir() else Path("../Part A")
out = Path("Part C"); out.mkdir(exist_ok=True)

or_table = pd.read_csv(part_a / "or_table_main_model.csv", index_col=0)
spec = json.loads((part_a / "model_spec.json").read_text(encoding="utf-8"))
train = pd.read_csv(part_a / "train_data.csv", index_col=0)
coef = pd.Series(spec["coefficients"])
mu_t = spec["centering"]["mean_log2_target"]
mu_d = spec["centering"]["mean_duration_10"]

INK, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
BLUE, RED, GRAY, ORANGE = "#2a78d6", "#e34948", "#a09f97", "#eb6834"
plt.rcParams.update({
    "font.size": 11, "axes.edgecolor": AXIS, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100,
})

def predict(goal_usd, duration_days, prev=0, images=1, videos=0, rewards=None, cat=None):
    """Predicted success probability from Part A's final coefficients."""
    rewards = train["Rewards"].median() if rewards is None else rewards
    t = np.log2(goal_usd) - mu_t
    d = duration_days / 10 - mu_d
    z = (coef["const"] + coef["Log2_Target_c"] * t + coef["Duration_10_c"] * d
         + coef["Target_x_Duration"] * t * d + coef["Prev_Success_Bool"] * prev
         + coef["Images_Bool"] * images + coef["Videos_Bool"] * videos
         + coef["Rewards"] * rewards + (coef[cat] if cat else 0))
    return 1 / (1 + np.exp(-z))

## 2. Odds-ratio forest plot

**Purpose.** Show every effect on one scale. Points to the left of 1 lower the odds of success, points to the right raise them; a CI crossing 1 means not significant.

In [ ]:
labels = {
    "Log2_Target_c": "Funding goal doubles (H1)",
    "Duration_10_c": "+10 days duration (H2)",
    "Target_x_Duration": "Goal × Duration (H3)",
    "Prev_Success_Bool": "Previous successful campaign (H4)",
    "Images_Bool": "Has images (H5)",
    "Videos_Bool": "Has video (H5)",
    "Rewards": "+1 reward tier (H5)",
    "Cat_Creative_Arts": "Category: Creative Arts",
    "Cat_Design": "Category: Design",
    "Cat_Food": "Category: Food",
    "Cat_Literature_Journalism": "Category: Literature & Journalism",
}
f = or_table.loc[list(labels)].iloc[::-1]
y = np.arange(len(f))
sig = f["p_value"] < 0.05
colors = np.where(~sig, GRAY, np.where(f["Odds_Ratio"] > 1, BLUE, RED))

fig, ax = plt.subplots(figsize=(9, 5.6))
ax.axvline(1, color=AXIS, lw=1.2, zorder=0)
ax.axhspan(-0.5, 3.5, color="#f4f3ef", zorder=-1)   # category controls band
ax.hlines(y, f["CI_Lower"], f["CI_Upper"], colors=colors, lw=2.2)
ax.scatter(f["Odds_Ratio"], y, s=55, c=colors, zorder=3, edgecolor="white", linewidth=1.5)
for yi, (_, r) in zip(y, f.iterrows()):
    ax.text(5.6, yi, f"{r['Odds_Ratio']:.2f}  [{r['CI_Lower']:.2f}, {r['CI_Upper']:.2f}]{r['Sig'] if isinstance(r['Sig'], str) else ''}",
            va="center", fontsize=9.5, color=MUTED)
ax.set_yticks(y, [labels[i] for i in f.index], color=INK)
ax.set_xscale("log")
ax.set_xlim(0.45, 5.4)
ax.xaxis.set_major_locator(mticker.FixedLocator([0.5, 0.75, 1, 1.5, 2, 3, 4]))
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%g"))
ax.xaxis.set_minor_locator(mticker.NullLocator())
ax.grid(axis="x", color=GRID, lw=0.8)
ax.set_xlabel("Odds ratio (log scale), 95% CI")
ax.set_ylim(-0.6, len(f) - 0.2)
fig.suptitle("Which pre-launch factors move the odds of success?", x=0.01, ha="left", fontsize=13, color=INK)
ax.text(5.6, len(f) - 0.45, "OR  [95% CI]", fontsize=9.5, color=INK, va="bottom", weight="bold")
fig.text(0.01, 0.01, "Blue = raises odds, red = lowers odds, gray = not significant (p ≥ 0.05). Categories vs Media & Entertainment.  *** p<0.001, ** p<0.01",
         fontsize=8.5, color=MUTED)
fig.subplots_adjust(left=0.33, right=0.78, top=0.88, bottom=0.14)
fig.savefig(out / "fig1_or_forest.png", dpi=200)

## 3. Goal × duration interaction (H3)

**Purpose.** Translate the interaction into predicted probabilities for goals at the 25th/50th/75th percentiles and durations of 15/30/45 days, for a typical first-time creator.

In [ ]:
goals = train["Target"].quantile([0.25, 0.50, 0.75])
days = np.linspace(15, 60, 100)
shades = ["#86b6ef", "#2a78d6", "#104281"]
names = ["25th pct goal", "Median goal", "75th pct goal"]

fig, ax = plt.subplots(figsize=(8.5, 5.2))
grid_rows = []
for g, c, n in zip(goals, shades, names):
    p = predict(g, days)
    ax.plot(days, p * 100, color=c, lw=2.2)
    pts = np.array([15, 30, 45])
    pp = predict(g, pts)
    ax.scatter(pts, pp * 100, color=c, s=50, zorder=3, edgecolor="white", linewidth=1.5)
    ax.text(61, p[-1] * 100, f"{n}\n(${g:,.0f})", color=INK, fontsize=9.5, va="center")
    for dd, v in zip(pts, pp):
        grid_rows.append({"Goal": n, "Goal_USD": round(g), "Duration_days": dd, "Pred_success_%": round(v * 100, 1)})
ax.set_xlim(14, 60); ax.set_xticks([15, 30, 45, 60])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.grid(axis="y", color=GRID, lw=0.8)
ax.set_xlabel("Campaign duration (days)"); ax.set_ylabel("Predicted probability of success")
ax.set_title("Longer campaigns hurt more when the goal is high (H3)", loc="left", fontsize=13, pad=12)
fig.text(0.01, 0.01, "Typical first-time creator: Media & Entertainment, images, no video, 7 reward tiers.\nThe interaction is significant in the main model but not robust to dropping extreme goals (Part D).",
         fontsize=8.5, color=MUTED)
fig.subplots_adjust(left=0.11, right=0.80, top=0.9, bottom=0.18)
fig.savefig(out / "fig2_interaction.png", dpi=200)
interaction_grid = pd.DataFrame(grid_rows).pivot(index="Goal", columns="Duration_days", values="Pred_success_%").loc[names]
interaction_grid.to_csv(out / "interaction_grid.csv")
print(interaction_grid)

## 4. Goal size by creator track record (H1 + H4)

**Purpose.** Show how much a previous success offsets a larger goal, as a basis for recommendations.

In [ ]:
g = np.logspace(2, 5, 200)
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for prev, c, n in [(1, BLUE, "Repeat creator\n(previous success)"), (0, ORANGE, "First-time\ncreator")]:
    p = predict(g, 30, prev=prev)
    ax.plot(g, p * 100, color=c, lw=2.2)
    ax.text(1.1e5, p[-1] * 100, n, color=INK, fontsize=9.5, va="center")
ax.set_xscale("log")
ax.xaxis.set_major_locator(mticker.FixedLocator([100, 1000, 10000, 100000]))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.xaxis.set_minor_locator(mticker.NullLocator())
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.set_ylim(0, 100)
ax.grid(axis="y", color=GRID, lw=0.8)
ax.set_xlabel("Funding goal (log scale)"); ax.set_ylabel("Predicted probability of success")
ax.set_title("A track record offsets a much bigger goal (H1 + H4)", loc="left", fontsize=13, pad=12)
fig.text(0.01, 0.01, "30-day campaign, Media & Entertainment, images, no video, 7 reward tiers.",
         fontsize=8.5, color=MUTED)
fig.subplots_adjust(left=0.11, right=0.80, top=0.9, bottom=0.15)
fig.savefig(out / "fig3_goal_by_creator.png", dpi=200)

## 5. H1–H5 verdict table

**Purpose.** Summarize whether each hypothesis is supported and how large the effect is.

In [ ]:
def pct(orv): return f"{(orv - 1) * 100:+.0f}%"
r = or_table
h = pd.DataFrame([
    ["H1", "Higher goal → lower success (−)", f"OR {r.loc['Log2_Target_c','Odds_Ratio']:.2f} per doubling ({pct(r.loc['Log2_Target_c','Odds_Ratio'])} odds), p<0.001", "Supported"],
    ["H2", "Longer duration → lower success (−)", f"OR {r.loc['Duration_10_c','Odds_Ratio']:.2f} per +10 days ({pct(r.loc['Duration_10_c','Odds_Ratio'])} odds), p<0.001", "Supported"],
    ["H3", "Duration penalty stronger for high goals", f"Interaction OR {r.loc['Target_x_Duration','Odds_Ratio']:.2f}, p=0.004; LR test p=0.004; not robust to extreme goals (Part D)", "Supported, not robust"],
    ["H4", "Previous success → higher success (+)", f"OR {r.loc['Prev_Success_Bool','Odds_Ratio']:.2f} ({pct(r.loc['Prev_Success_Bool','Odds_Ratio'])} odds), p<0.001", "Supported"],
    ["H5", "Richer presentation → higher success (+)",
     f"Rewards OR {r.loc['Rewards','Odds_Ratio']:.2f} per tier (p<0.001); images p={r.loc['Images_Bool','p_value']:.2f}, video p={r.loc['Videos_Bool','p_value']:.2f}",
     "Partly supported"],
], columns=["Hypothesis", "Prediction", "Evidence", "Verdict"])
h.to_csv(out / "hypothesis_table.csv", index=False)
print(h.to_string(index=False))

short = [
    ["H1", "Higher goal → lower success", "OR 0.62 per doubling of goal (p<0.001)"],
    ["H2", "Longer duration → lower success", "OR 0.85 per +10 days (p<0.001)"],
    ["H3", "Duration hurts more for high goals", "OR 0.97 (p=0.004); p=0.43 without extreme goals"],
    ["H4", "Previous success → higher success", "OR 3.14 (p<0.001)"],
    ["H5", "Richer page → higher success", "Reward tiers OR 1.10 (p<0.001); images, video n.s."],
]
xs = [0.0, 0.06, 0.40, 0.84]
fig, ax = plt.subplots(figsize=(11, 3.3)); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 6.2)
for x, t in zip(xs, ["", "Hypothesis", "Evidence (odds ratio)", "Verdict"]):
    ax.text(x, 5.6, t, weight="bold", color=INK, fontsize=11, va="center")
ax.plot([0, 1], [5.2, 5.2], color=AXIS, lw=1.2)
for i, (row, verdict) in enumerate(zip(short, h["Verdict"])):
    yy = 4.6 - i
    for x, t in zip(xs, row):
        ax.text(x, yy, t, color=INK if x < 0.4 else MUTED, fontsize=11, va="center", weight="bold" if x == 0 else "normal")
    ok = verdict == "Supported"
    ax.text(xs[3], yy, ("✓ " if ok else "◐ ") + verdict.replace("Supported, not robust", "Fragile"), color="#006300" if ok else "#9a5b00", fontsize=11, va="center", weight="bold")
    if i < 4: ax.plot([0, 1], [yy - 0.5, yy - 0.5], color=GRID, lw=0.8)
fig.savefig(out / "fig4_hypothesis_table.png", dpi=200, bbox_inches="tight")

## 6. Numbers quoted in the slides

**Purpose.** Compute the specific probabilities and odds multipliers used in the interpretation and recommendations.

In [ ]:
dur_or = lambda goal: np.exp(coef["Duration_10_c"] + coef["Target_x_Duration"] * (np.log2(goal) - mu_t))
for n, gv in zip(names, goals):
    print(f"{n} (${gv:,.0f}): +10 days multiplies odds by {dur_or(gv):.3f}")
print("Repeat vs first-time at $3,000 / 30 days:", round(predict(3000, 30, prev=1)*100, 1), round(predict(3000, 30)*100, 1))
for gv in [1000, 5000, 10000, 20000, 50000]:
    print(f"goal ${gv:,}: first-time {predict(gv,30)*100:.1f}%  repeat {predict(gv,30,prev=1)*100:.1f}%")
print("Rewards 3 vs 10 tiers at median goal:", round(predict(2961,30,rewards=3)*100,1), round(predict(2961,30,rewards=10)*100,1))